# Lab 07 · Token counting and why migrations need re-measuring
**~20 minutes · costs about $0.001 (count_tokens is free) · Domain 5 (16.8%)**

Token counts are **model-specific**. They do not transfer across a migration,
and newer tokenizers can produce meaningfully more tokens for identical text.

In [ ]:
import os, anthropic
client = anthropic.Anthropic()          # reads ANTHROPIC_API_KEY
MODEL  = "claude-sonnet-4-6"            # verify against lab 00 output
CHEAP  = "claude-haiku-4-5"             # for high-volume steps
print("sdk", anthropic.__version__)

## The same text, measured against several models

In [ ]:
SAMPLE = """def reconcile(ledger, statements, tolerance=0.01):
    unmatched = []
    for entry in ledger:
        hit = next((s for s in statements if abs(s.amount - entry.amount) < tolerance), None)
        if hit is None:
            unmatched.append(entry)
    return unmatched"""

ids = [m.id for m in client.models.list()][:6]
base = None
for mid in ids:
    try:
        n = client.messages.count_tokens(model=mid,
                messages=[{"role":"user","content":SAMPLE}]).input_tokens
        if base is None: base = n
        print(f"{mid:<34} {n:>6} tokens   {n/base:>5.2f}x")
    except Exception as e:
        print(f"{mid:<34} n/a ({type(e).__name__})")

## Count the whole request, not just the message

Tools and the system prompt are input tokens too. This is the number your cost
model needs.

In [ ]:
TOOLS = [{"name":"search","description":"Search the knowledge base for relevant passages.",
          "input_schema":{"type":"object","properties":{"q":{"type":"string"}},"required":["q"]}}]
SYS = "You are a meticulous financial reconciliation assistant." * 8

for label, kw in [
    ("message only",        dict(messages=[{"role":"user","content":SAMPLE}])),
    ("+ system",            dict(system=SYS, messages=[{"role":"user","content":SAMPLE}])),
    ("+ system + tools",    dict(system=SYS, tools=TOOLS, messages=[{"role":"user","content":SAMPLE}])),
]:
    print(f"{label:<20}", client.messages.count_tokens(model=MODEL, **kw).input_tokens)

## Context window is not max output

Two different ceilings. The exam tests that you know it.

In [ ]:
for m in client.models.list():
    mi, mo = getattr(m,"max_input_tokens",None), getattr(m,"max_tokens",None)
    if mi: print(f"{m.id:<34} context={mi:<10} max_output={mo}")

## Effort, and the parameters that now 400

On current models `effort` replaced sampling parameters. Confirm it yourself
rather than trusting my table.

In [ ]:
import anthropic
for attempt, kw in [
    ("effort=low",        dict(effort="low")),
    ("temperature=0.2",   dict(temperature=0.2)),
    ("manual thinking",   dict(thinking={"type":"enabled","budget_tokens":2000})),
]:
    try:
        r = client.messages.create(model=MODEL, max_tokens=200,
                messages=[{"role":"user","content":"What is 17*23? Answer only."}], **kw)
        print(f"{attempt:<20} OK   out={r.usage.output_tokens}")
    except anthropic.APIStatusError as e:
        print(f"{attempt:<20} {e.status_code}  {e.message[:90]}")
    except TypeError as e:
        print(f"{attempt:<20} sdk rejected: {e}")

---
### Checkpoint
- Why must you re-run token counting after a migration?
- Context window vs max output tokens?
- What replaced temperature as the control surface, and is it a token cap?